In [ ]:
!pip install sentence-transformers rouge-score

In [ ]:
!pip install bert-score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install sympy==1.12 --force-reinstall
!pip install torch --force-reinstall

In [ ]:
import re
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer

# =========================
# 0. FILE CONFIG (KONSTANTA)
# =========================
INPUT_FILE_PATH = r"/content/drive/MyDrive/gemini/TEXT/dataset_anonim_phase4_.xlsx"
OUTPUT_FILE_PATH = r"/content/drive/MyDrive/gemini/TEXT/dataset_anonim_phase4_evaluation.xlsx"

# =========================
# 1. PREPROCESSING
# =========================
def preprocess(text):
    if pd.isna(text):
        return []
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

# =========================
# 2. OVERLAP COEFFICIENT
# =========================
def overlap_coefficient(raw, extracted):
    raw_tokens = set(preprocess(raw))
    ext_tokens = set(preprocess(extracted))
    intersection = raw_tokens.intersection(ext_tokens)
    if min(len(raw_tokens), len(ext_tokens)) == 0:
        return 0.0
    return len(intersection) / min(len(raw_tokens), len(ext_tokens))

# =========================
# 3. COSINE SIMILARITY
# =========================
def cosine_similarity(raw, extracted, model):
    emb_raw = model.encode(str(raw), convert_to_tensor=True)
    emb_ext = model.encode(str(extracted), convert_to_tensor=True)
    cosine = util.cos_sim(emb_raw, emb_ext)
    return cosine.item()

# =========================
# 4. ROUGE
# =========================
def rouge1_recall(raw, extracted):
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)
    scores = scorer.score(str(raw), str(extracted))
    return scores['rouge1'].recall

def rougeL_all(raw, extracted):
    raw_tokens = str(raw).split()
    ext_tokens = str(extracted).split()
    m, n = len(raw_tokens), len(ext_tokens)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,n+1):
            if raw_tokens[i-1] == ext_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1]+1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs_len = dp[m][n]

    recall = lcs_len / len(ext_tokens) if len(ext_tokens)>0 else 0.0
    precision = lcs_len / len(raw_tokens) if len(raw_tokens)>0 else 0.0
    f1 = 2*recall*precision/(recall+precision) if (recall+precision)>0 else 0.0

    return recall, precision, f1

# =========================
# 5. HARMONIC MEAN
# =========================
def harmonic_mean(values):
    if any(v == 0 for v in values):
        return 0.0
    return len(values) / sum(1/v for v in values)

# =========================
# 6. MANUAL BERTSCORE
# =========================
def manual_bertscore(raw, extracted, model):
    raw_tokens = str(raw).split()
    ext_tokens = str(extracted).split()
    if len(raw_tokens)==0 or len(ext_tokens)==0:
        return 0.0
    emb_raw = model.encode(raw_tokens, convert_to_tensor=True)
    emb_ext = model.encode(ext_tokens, convert_to_tensor=True)
    sim_matrix = util.cos_sim(emb_ext, emb_raw)
    max_sim_per_token = sim_matrix.max(dim=1).values
    return max_sim_per_token.mean().item()

# =========================
# 7. LOAD MODEL (sekali saja)
# =========================
model = SentenceTransformer("indobenchmark/indobert-large-p2")

# =========================
# 8. LOAD EXCEL
# =========================
df = pd.read_excel(INPUT_FILE_PATH)

# Pastikan kolom ada
assert 'question' in df.columns
assert 'paraphrase' in df.columns

# =========================
# 9. LOOP & HITUNG
# =========================
results_list = []

for idx, row in df.iterrows():

    raw = row['question']
    extracted = row['paraphrase']

    overlap = overlap_coefficient(raw, extracted)
    cosine = cosine_similarity(raw, extracted, model)
    r1 = rouge1_recall(raw, extracted)
    rL_recall, rL_precision, rL_f1 = rougeL_all(raw, extracted)
    bert = manual_bertscore(raw, extracted, model)

    extraction_score_weight = 0.3*overlap + 0.3*cosine + 0.2*r1 + 0.2*bert
    extraction_score_harmonic = harmonic_mean([overlap, cosine, r1, bert])

    metrics = {
        "overlap": overlap,
        "cosine_similarity": cosine,
        "rouge1_recall": r1,
        "rougeL_recall": rL_recall,
        "rougeL_precision": rL_precision,
        "rougeL_f1": rL_f1,
        "bertscore_f1": bert
    }

    # SUBSET SCORE MEAN
    lexical = (overlap + r1 + rL_f1) / 3
    semantic = (cosine + bert) / 2
    subset_mean = 0.5*lexical + 0.5*semantic

    # SUBSET SCORE HARMONIC
    lexical_h = harmonic_mean([overlap, r1, rL_f1])
    semantic_h = harmonic_mean([cosine, bert])
    subset_harmonic = 0.5*lexical_h + 0.5*semantic_h

    results_list.append({
        "rouge1_recall": round(r1,4),
        "overlap": round(overlap,4),
        "cosine_similarity": round(cosine,4),
        "rougeL_recall": round(rL_recall,4),
        "rougeL_precision": round(rL_precision,4),
        "rougeL_f1": round(rL_f1,4),
        "bertscore_f1": round(bert,4),
        "extraction_score_weight": round(extraction_score_weight,4),
        "extraction_score_harmonic": round(extraction_score_harmonic,4),
        "subset_score_mean": round(subset_mean,4),
        "subset_score_harmonic_mean": round(subset_harmonic,4)
    })

# =========================
# 10. GABUNGKAN KE DATAFRAME
# =========================
results_df = pd.DataFrame(results_list)
final_df = pd.concat([df, results_df], axis=1)

# =========================
# 11. SAVE KE XLSX
# =========================
final_df.to_excel(OUTPUT_FILE_PATH, index=False)

print("Proses selesai.")
print("Output disimpan di:", OUTPUT_FILE_PATH)